<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/EVO2_TOPOAI_TRANSFORMER_MULTRUN_CORRECTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://huggingface.co/arcinstitute/evo2_40b_base


https://www.nytimes.com/2026/08/06/science/ai-viruses-bacteria-arc.html?campaign_id=34&emc=edit_sc_20260807&instance_id=179957&nl=science-times&regi_id=174448907&segment_id=224332&user_id=e172a1289d72c2a21d5b77fdac027238

In [1]:
!nvidia-smi

Fri Aug  7 22:07:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [4]:
!pip install evo2 --no-build-isolation -q

!pip install https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl -q

In [ ]:
![]

In [22]:
!pip show evo2 flash_attn| egrep "Name|Version:"

Name: evo2
Version: 0.6.0
Name: flash_attn
Version: 2.8.3


## EVO2

In [23]:
import torch
from evo2 import Evo2

from warnings import filterwarnings
filterwarnings('ignore')

class Evo2TopologicalGovernorWrapper:
    """
    Interfaces with the official Evo2 manager to intercept layer activations
    and apply Arithmetic Spectral Theory (AST) topological anchors.
    """
    def __init__(self, model_name: str = 'evo2_7b', tolerance: float = 1e-4):
        print(f"Loading official {model_name} architecture manager...")
        # Initialize the official Evo2 model manager
        self.evo2_manager = Evo2(model_name)

        # Extract the underlying PyTorch model from the manager handle
        self.model = getattr(self.evo2_manager, 'model', self.evo2_manager)
        self.device = next(self.model.parameters()).device

        self.tolerance = tolerance
        self.anchors = {}
        self.hooks = []

    def _get_governance_hook(self, layer_name: str):
        """Creates a forward hook to anchor internal layer states dynamically."""
        def hook(module, input, output):
            # Handle tuple outputs common in hybrid architectures
            is_tuple = isinstance(output, tuple)
            hidden_states = output[0] if is_tuple else output

            if layer_name not in self.anchors:
                # Capture baseline topological manifold from initial pass
                self.anchors[layer_name] = hidden_states.detach().clone()
                print(f"Topological anchor registered for: {layer_name}")
            else:
                # Enforce spectral constraint against weight/state drift
                anchor = self.anchors[layer_name]
                if anchor.shape == hidden_states.shape:
                    deviation = torch.abs(hidden_states - anchor)
                    mask = (deviation > self.tolerance).float()
                    governed_states = hidden_states * (1 - mask) + anchor * mask

                    if is_tuple:
                        output = (governed_states,) + output[1:]
                    else:
                        output = governed_states
            return output
        return hook

    def register_governance_hook(self, layer_path: str = 'blocks.28'):
        """Attaches the TOPO governor to a targeted block within the model."""
        for name, module in self.model.named_modules():
            if name == layer_path:
                handle = module.register_forward_hook(self._get_governance_hook(layer_path))
                self.hooks.append(handle)
                print(f"TOPO governance hook successfully bound to layer: {layer_path}")
                break

    def generate_governed_sequence(self, prompt: str, n_tokens: int = 50):
        """Runs generation through Evo2 with active topological governance."""
        input_seqs = [prompt]
        # Use the native generation pipeline of the manager
        output = self.evo2_manager.generate(
            input_seqs,
            n_tokens=n_tokens,
            temperature=0.7,
            top_k=4
        )
        return output

if __name__ == "__main__":
    # Instantiate the wrapper
    governed_wrapper = Evo2TopologicalGovernorWrapper('evo2_7b')

    # Bind the topological governor to a deep convolutional/hybrid block layer
    governed_wrapper.register_governance_hook('blocks.28')

    # Run a test genomic sequence generation
    test_prompt = "ACGTTGCAATGCGATCG"
    generated_output = governed_wrapper.generate_governed_sequence(test_prompt, n_tokens=30)

    print("\nGenerated Governed Sequence Result:")
    print(generated_output)

Loading official evo2_7b architecture manager...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt




  0%|          | 0/32 [00:00<?, ?it/s]

 12%|█▎        | 4/32 [00:00<00:01, 16.07it/s]

100%|██████████| 32/32 [00:00<00:00, 86.05it/s]


Extra keys in state_dict: {'blocks.20.projections._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.3.mixer.dense._extra_state', 'blocks.15.projections._extra_state', 'blocks.18.projections._extra_state', 'blocks.1.projections._extra_state', 'blocks.28.projections._extra_state', 'blocks.10.mixer.attn._extra_state', 'blocks.26.projections._extra_state', 'blocks.2.mixer.mixer.filter.t', 'blocks.9.mixer.mixer.filter.t', 'blocks.12.projections._extra_state', 'blocks.17.mixer.dense._extra_state', 'blocks.24.mixer.dense._extra_state', 'unembed.weight', 'blocks.10.mixer.dense._extra_state', 'blocks.8.projections._extra_state', 'blocks.5.projections._extra_state', 'blocks.30.mixer.mixer.filter.t', 'blocks.7.projections._extra_state', 'blocks.2.projections._extra_state', 'blocks.6.mixer.mixer.filter.t', 'blocks.14.projections._extra_state', 'blocks.16.mixer.mixer.filter.t', 'blocks.25.projections._extra_state', 'blocks.30.projections._extra_state', 'blocks.11.projections._extra_state', '

In [1]:
import os
import torch
import torch.nn as nn
from torch.optim import AdamW
from evo2 import Evo2
from warnings import filterwarnings
filterwarnings('ignore')

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

class GovernedEvo2Trainer:
    def __init__(self, model_name: str = 'evo2_7b'):
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        print(f"Initializing official Evo 2 Manager for {model_name} on {self.device}...")

        self.evo2_manager = Evo2(model_name)
        self.model = getattr(self.evo2_manager, 'model', self.evo2_manager)

        # Convert inference parameters into standard independent mutable parameters
        self._convert_to_standard_parameters()

        self.anchors = {}
        self.captured_states = None
        self._register_clean_hook('blocks.28')

    def _convert_to_standard_parameters(self):
        """Strips inference flags by rebuilding parameters as standard leaf nodes."""
        for name, module in self.model.named_modules():
            for param_name, param in module.named_parameters(recurse=False):
                if param is not None:
                    # Clone and detach into a fresh standard parameter tensor
                    new_param = nn.Parameter(param.data.clone().detach(), requires_grad=True)
                    setattr(module, param_name, new_param)

    def _register_clean_hook(self, target_layer: str):
        def hook(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            normal_states = hidden_states.detach().clone().requires_grad_(True)
            if target_layer not in self.anchors:
                self.anchors[target_layer] = normal_states.detach().clone()
                print(f"TOPO Baseline Anchor successfully locked for: {target_layer}")
            self.captured_states = normal_states

        for name, module in self.model.named_modules():
            if name == target_layer:
                module.register_forward_hook(hook)
                break

    def fine_tune_step(self, sequence_str: str, optimizer: AdamW, criterion: nn.Module, lambda_topo: float = 0.1):
        with torch.set_grad_enabled(True):
            self.model.train()
            optimizer.zero_grad()

            token_ids = self.evo2_manager.tokenizer.tokenize(sequence_str)
            input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(self.device)

            outputs = self.model(input_ids)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()

            task_loss = criterion(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

            topo_loss = torch.tensor(0.0, device=self.device, requires_grad=True)
            for layer_name, anchor in self.anchors.items():
                if self.captured_states is not None and anchor.shape == self.captured_states.shape:
                    deviation = torch.norm(self.captured_states - anchor, p=2, dim=-1)
                    topo_loss = lambda_topo * torch.mean(torch.relu(deviation - 1e-4))

            total_loss = task_loss + topo_loss
            total_loss.backward()
            optimizer.step()

            return task_loss.item(), topo_loss.item()

if __name__ == "__main__":
    trainer = GovernedEvo2Trainer('evo2_7b')
    optimizer = AdamW(trainer.model.parameters(), lr=1e-5)
    criterion = nn.CrossEntropyLoss()

    genomic_sample = "ACGTTGCAATGCGATCGATCGATCGATCGATCGATCGATC"

    task_l, topo_l = trainer.fine_tune_step(genomic_sample, optimizer, criterion)
    print(f"Governed Fine-Tuning Step Successful -> Task Loss: {task_l:.4f} | TOPO Invariant Penalty: {topo_l:.4f}")

Initializing official Evo 2 Manager for evo2_7b on cuda:0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt




  0%|          | 0/32 [00:00<?, ?it/s]

 12%|█▎        | 4/32 [00:00<00:00, 34.18it/s]

100%|██████████| 32/32 [00:00<00:00, 116.70it/s]


Extra keys in state_dict: {'blocks.17.mixer.dense._extra_state', 'blocks.7.projections._extra_state', 'blocks.13.mixer.mixer.filter.t', 'blocks.30.mixer.mixer.filter.t', 'blocks.23.projections._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.0.projections._extra_state', 'blocks.6.mixer.mixer.filter.t', 'blocks.29.projections._extra_state', 'blocks.23.mixer.mixer.filter.t', 'blocks.17.mixer.attn._extra_state', 'blocks.24.mixer.dense._extra_state', 'blocks.13.projections._extra_state', 'blocks.19.projections._extra_state', 'blocks.21.projections._extra_state', 'blocks.30.projections._extra_state', 'blocks.31.mixer.attn._extra_state', 'blocks.12.projections._extra_state', 'unembed.weight', 'blocks.16.mixer.mixer.filter.t', 'blocks.8.projections._extra_state', 'blocks.18.projections._extra_state', 'blocks.15.projections._extra_state', 'blocks.10.mixer.dense._extra_state', 'blocks.3.mixer.dense._extra_state', 'blocks.2.mixer.mixer.filter.t', 'blocks.26.projections._extra_state', 'bl

## EVO2-TOPO

In [2]:
import os
import torch
import torch.nn as nn
from torch.optim import AdamW
from evo2 import Evo2

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

class GovernedEvo2Trainer:
    """
    Complete production trainer and inference wrapper integrating
    TOPO structural governance into Evo 2 to prevent catastrophic forgetting.
    """
    def __init__(self, model_name: str = 'evo2_7b'):
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        print(f"Initializing official Evo 2 Manager for {model_name} on {self.device}...")

        self.evo2_manager = Evo2(model_name)
        self.model = getattr(self.evo2_manager, 'model', self.evo2_manager)

        # Convert model parameters into standard independent mutable parameters
        self._convert_to_standard_parameters()

        self.anchors = {}
        self.captured_states = None
        self._register_clean_hook('blocks.28')

    def _convert_to_standard_parameters(self):
        for name, module in self.model.named_modules():
            for param_name, param in module.named_parameters(recurse=False):
                if param is not None:
                    new_param = nn.Parameter(param.data.clone().detach(), requires_grad=True)
                    setattr(module, param_name, new_param)

    def _register_clean_hook(self, target_layer: str):
        def hook(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            normal_states = hidden_states.detach().clone().requires_grad_(True)
            if target_layer not in self.anchors:
                self.anchors[target_layer] = normal_states.detach().clone()
                print(f"TOPO Baseline Anchor successfully locked for: {target_layer}")
            self.captured_states = normal_states

        for name, module in self.model.named_modules():
            if name == target_layer:
                module.register_forward_hook(hook)
                break

    def fine_tune_step(self, sequence_str: str, optimizer: AdamW, criterion: nn.Module, lambda_topo: float = 0.1):
        with torch.set_grad_enabled(True):
            self.model.train()
            optimizer.zero_grad()

            token_ids = self.evo2_manager.tokenizer.tokenize(sequence_str)
            input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(self.device)

            outputs = self.model(input_ids)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()

            task_loss = criterion(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

            topo_loss = torch.tensor(0.0, device=self.device, requires_grad=True)
            for layer_name, anchor in self.anchors.items():
                if self.captured_states is not None and anchor.shape == self.captured_states.shape:
                    deviation = torch.norm(self.captured_states - anchor, p=2, dim=-1)
                    topo_loss = lambda_topo * torch.mean(torch.relu(deviation - 1e-4))

            total_loss = task_loss + topo_loss
            total_loss.backward()
            optimizer.step()

            return task_loss.item(), topo_loss.item()

    def save_checkpoint(self, save_directory: str = "./governed_evo2_checkpoint"):
        os.makedirs(save_directory, exist_ok=True)
        checkpoint_payload = {
            "model_state_dict": self.model.state_dict(),
            "topo_anchors": self.anchors,
        }
        output_path = os.path.join(save_directory, "governed_evo2_state.pt")
        torch.save(checkpoint_payload, output_path)
        print(f"Governed checkpoint successfully saved to: {output_path}")

    def load_checkpoint(self, load_path: str = "./governed_evo2_checkpoint/governed_evo2_state.pt"):
        checkpoint = torch.load(load_path, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.anchors = checkpoint["topo_anchors"]
        print(f"Governed model weights and TOPO anchors successfully loaded from: {load_path}")



In [2]:
# --- CELL 1: TRAINING & SAVING PIPELINE ---
if __name__ == "__main__":
    trainer = GovernedEvo2Trainer('evo2_7b')
    optimizer = AdamW(trainer.model.parameters(), lr=1e-5)
    criterion = nn.CrossEntropyLoss()

    genomic_sample = "ACGTTGCAATGCGATCGATCGATCGATCGATCGATCGATC"

    task_l, topo_l = trainer.fine_tune_step(genomic_sample, optimizer, criterion)
    print(f"Fine-Tuning Step Successful -> Task Loss: {task_l:.4f} | TOPO Invariant Penalty: {topo_l:.4f}")

    # Save the governed state and anchors
    trainer.save_checkpoint("./governed_evo2_checkpoint")

Initializing official Evo 2 Manager for evo2_7b on cuda:0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt


/usr/local/lib/python3.12/dist-packages/evo2/models.py:294: UserWarning: Transformer Engine not installed. Falling back to bf16 projections (use_fp8_input_projections=False). 
  warnings.warn(


  0%|          | 0/32 [00:00<?, ?it/s]

 12%|█▎        | 4/32 [00:00<00:00, 34.06it/s]

100%|██████████| 32/32 [00:00<00:00, 110.73it/s]


Extra keys in state_dict: {'blocks.4.projections._extra_state', 'blocks.27.projections._extra_state', 'blocks.31.mixer.attn._extra_state', 'blocks.8.projections._extra_state', 'blocks.26.projections._extra_state', 'blocks.3.mixer.attn._extra_state', 'blocks.29.projections._extra_state', 'blocks.15.projections._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.27.mixer.mixer.filter.t', 'blocks.9.mixer.mixer.filter.t', 'blocks.7.projections._extra_state', 'blocks.28.projections._extra_state', 'blocks.23.mixer.mixer.filter.t', 'blocks.25.projections._extra_state', 'blocks.24.mixer.dense._extra_state', 'blocks.23.projections._extra_state', 'blocks.12.projections._extra_state', 'blocks.30.projections._extra_state', 'blocks.10.mixer.attn._extra_state', 'blocks.24.mixer.attn._extra_state', 'blocks.19.projections._extra_state', 'blocks.17.mixer.dense._extra_state', 'blocks.20.projections._extra_state', 'blocks.18.projections._extra_state', 'blocks.2.mixer.mixer.filter.t', 'blocks.1.proje

In [3]:
# --- CELL 2: INFERENCE & RELOADING PIPELINE ---
from warnings import filterwarnings
filterwarnings('ignore')

if __name__ == "__main__":
    # Initialize a fresh instance or load into existing
    inference_trainer = GovernedEvo2Trainer('evo2_7b')

    # Load previously trained governed weights and TOPO invariant anchors
    inference_trainer.load_checkpoint("./governed_evo2_checkpoint/governed_evo2_state.pt")

    # Run generation using the governed model manager
    test_prompt = "ACGTTGCAATGCGATCG"
    input_seqs = [test_prompt]

    generated_output = inference_trainer.evo2_manager.generate(
        input_seqs,
        n_tokens=30,
        temperature=0.7,
        top_k=4
    )

    print("\nInference Output with Governed Weights:")
    print(generated_output)

Initializing official Evo 2 Manager for evo2_7b on cuda:0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt




  0%|          | 0/32 [00:00<?, ?it/s]

 12%|█▎        | 4/32 [00:00<00:00, 36.77it/s]

100%|██████████| 32/32 [00:00<00:00, 126.62it/s]


Extra keys in state_dict: {'blocks.21.projections._extra_state', 'blocks.27.mixer.mixer.filter.t', 'blocks.6.projections._extra_state', 'blocks.31.mixer.attn._extra_state', 'blocks.22.projections._extra_state', 'blocks.10.mixer.attn._extra_state', 'blocks.7.projections._extra_state', 'blocks.8.projections._extra_state', 'blocks.10.mixer.dense._extra_state', 'blocks.23.mixer.mixer.filter.t', 'blocks.16.mixer.mixer.filter.t', 'blocks.17.mixer.attn._extra_state', 'blocks.9.projections._extra_state', 'blocks.6.mixer.mixer.filter.t', 'blocks.23.projections._extra_state', 'blocks.13.mixer.mixer.filter.t', 'blocks.18.projections._extra_state', 'blocks.24.mixer.attn._extra_state', 'blocks.3.mixer.dense._extra_state', 'blocks.16.projections._extra_state', 'blocks.19.projections._extra_state', 'blocks.9.mixer.mixer.filter.t', 'blocks.5.projections._extra_state', 'blocks.30.mixer.mixer.filter.t', 'blocks.29.projections._extra_state', 'blocks.28.projections._extra_state', 'blocks.12.projections._e

## TOPO-CERTIFICATION

In [1]:
import os
import torch
import torch.nn as nn
from torch.optim import AdamW
from evo2 import Evo2

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

class GovernedEvo2Trainer:
    def __init__(self, model_name: str = 'evo2_7b'):
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        print(f"Initializing official Evo 2 Manager for {model_name} on {self.device}...")

        self.evo2_manager = Evo2(model_name)
        self.model = getattr(self.evo2_manager, 'model', self.evo2_manager)

        self._convert_to_standard_parameters()
        self.anchors = {}
        self.captured_states = None
        self._register_clean_hook('blocks.28')

    def _convert_to_standard_parameters(self):
        for name, module in self.model.named_modules():
            for param_name, param in module.named_parameters(recurse=False):
                if param is not None:
                    new_param = nn.Parameter(param.data.clone().detach(), requires_grad=True)
                    setattr(module, param_name, new_param)

    def _register_clean_hook(self, target_layer: str):
        def hook(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            normal_states = hidden_states.detach().clone().requires_grad_(True)
            if target_layer not in self.anchors:
                self.anchors[target_layer] = normal_states.detach().clone()
                print(f"TOPO Baseline Anchor successfully locked for: {target_layer}")
            self.captured_states = normal_states

        for name, module in self.model.named_modules():
            if name == target_layer:
                module.register_forward_hook(hook)
                break

    def fine_tune_step(self, sequence_str: str, optimizer: AdamW, criterion: nn.Module, lambda_topo: float = 0.1):
        with torch.set_grad_enabled(True):
            self.model.train()
            optimizer.zero_grad()

            token_ids = self.evo2_manager.tokenizer.tokenize(sequence_str)
            input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(self.device)

            outputs = self.model(input_ids)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()

            task_loss = criterion(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

            topo_loss = torch.tensor(0.0, device=self.device, requires_grad=True)
            for layer_name, anchor in self.anchors.items():
                if self.captured_states is not None and anchor.shape == self.captured_states.shape:
                    deviation = torch.norm(self.captured_states - anchor, p=2, dim=-1)
                    topo_loss = lambda_topo * torch.mean(torch.relu(deviation - 1e-4))

            total_loss = task_loss + topo_loss
            total_loss.backward()
            optimizer.step()

            return task_loss.item(), topo_loss.item()

def measure_catastrophic_forgetting(trainer, baseline_sequences: list, adapted_sequences: list, criterion=nn.CrossEntropyLoss()):
    print("\n--- Starting Catastrophic Forgetting Evaluation ---")
    trainer.model.eval()

    # 1. Measure pre-adaptation baseline performance
    baseline_losses = []
    with torch.no_grad():
        for seq in baseline_sequences:
            token_ids = trainer.evo2_manager.tokenizer.tokenize(seq)
            input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(trainer.device)

            outputs = trainer.model(input_ids)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()

            loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            baseline_losses.append(loss.item())

    mean_baseline_loss = sum(baseline_losses) / len(baseline_losses)
    print(f"Pre-Adaptation Baseline Loss: {mean_baseline_loss:.4f}")

    # 2. Fine-tune on new domain sequences with TOPO protection active
    optimizer = AdamW(trainer.model.parameters(), lr=1e-5)
    print("Fine-tuning model on new domain data...")
    for epoch in range(2):
        for seq in adapted_sequences:
            trainer.fine_tune_step(seq, optimizer, criterion, lambda_topo=0.1)

    # 3. Measure post-adaptation performance on the exact same baseline sequences
    trainer.model.eval()
    post_losses = []
    with torch.no_grad():
        for seq in baseline_sequences:
            token_ids = trainer.evo2_manager.tokenizer.tokenize(seq)
            input_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(trainer.device)

            outputs = trainer.model(input_ids)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous()

            loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            post_losses.append(loss.item())

    mean_post_loss = sum(post_losses) / len(post_losses)
    print(f"Post-Adaptation Baseline Loss: {mean_post_loss:.4f}")

    # 4. Compute Forgetting Percentage
    forgetting_percentage = ((mean_post_loss - mean_baseline_loss) / mean_baseline_loss) * 100
    print(f"Catastrophic Forgetting Rate: {forgetting_percentage:.2f}%")

    return forgetting_percentage

if __name__ == "__main__":
    # Initialize trainer
    trainer = GovernedEvo2Trainer('evo2_7b')

    # Define test data
    reference_baseline = [
        "ACGTTGCAATGCGATCGATCGATCGATCGATCGATCGATC",
        "GGCCATCGATCGATCGATCGATCGATCGATCGATCGATCG"
    ]
    new_domain_data = [
        "TTTTAAAAGGGCCCGGGAAATTTTCCCGGGAAATTTTGGG",
        "ATATATATATCGATCGATCGATCGATCGATCGATCGATCG"
    ]

    # Run evaluation and print results
    forgetting_rate = measure_catastrophic_forgetting(trainer, reference_baseline, new_domain_data)

Initializing official Evo 2 Manager for evo2_7b on cuda:0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt


/usr/local/lib/python3.12/dist-packages/evo2/models.py:294: UserWarning: Transformer Engine not installed. Falling back to bf16 projections (use_fp8_input_projections=False). 
  warnings.warn(


  0%|          | 0/32 [00:00<?, ?it/s]

 12%|█▎        | 4/32 [00:00<00:00, 38.00it/s]

100%|██████████| 32/32 [00:00<00:00, 127.66it/s]


Extra keys in state_dict: {'blocks.23.projections._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.2.projections._extra_state', 'blocks.20.projections._extra_state', 'blocks.7.projections._extra_state', 'blocks.15.projections._extra_state', 'blocks.19.projections._extra_state', 'blocks.26.projections._extra_state', 'blocks.13.projections._extra_state', 'blocks.23.mixer.mixer.filter.t', 'blocks.24.mixer.dense._extra_state', 'blocks.29.projections._extra_state', 'blocks.30.projections._extra_state', 'blocks.12.projections._extra_state', 'blocks.28.projections._extra_state', 'blocks.3.mixer.dense._extra_state', 'blocks.16.mixer.mixer.filter.t', 'blocks.25.projections._extra_state', 'blocks.17.mixer.attn._extra_state', 'blocks.9.mixer.mixer.filter.t', 'blocks.16.projections._extra_state', 'blocks.18.projections._extra_state', 'blocks.24.mixer.attn._extra_state', 'blocks.27.mixer.mixer.filter.t', 'blocks.13.mixer.mixer.filter.t', 'blocks.0.projections._extra_state', 'blocks.9.projec

In [2]:
import os
import torch

def export_governed_evo2_artifact(trainer, output_dir: str = "./governed_evo2_final_model"):
    """
    Serializes the fine-tuned Evo 2 model weights, optimizer state,
    and TOPO topological invariant anchors into a portable format.
    """
    os.makedirs(output_dir, exist_ok=True)

    artifact = {
        "model_state_dict": trainer.model.state_dict(),
        "topo_anchors": trainer.anchors,
        "model_config": getattr(trainer.evo2_manager, 'config', None)
    }

    save_path = os.path.join(output_dir, "governed_evo2_weights.pt")
    torch.save(artifact, save_path)
    print(f"Governed model artifact successfully exported to: {save_path}")

if __name__ == "__main__":
    # Export your trained trainer instance after fine-tuning
    export_governed_evo2_artifact(trainer)

Governed model artifact successfully exported to: ./governed_evo2_final_model/governed_evo2_weights.pt


In [3]:
from huggingface_hub import HfApi, create_repo
from google.colab import userdata

# Retrieve your token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')
username = 'frankmorales2020'
repo_id = f"{username}/governed-evo2-7b-topo"
weights_path = "./governed_evo2_final_model/governed_evo2_weights.pt"

api = HfApi(token=HF_TOKEN)

print(f"Ensuring repository exists: {repo_id}")
create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=HF_TOKEN)

print(f"Uploading governed artifact {weights_path} to Hugging Face Hub...")
api.upload_file(
    path_or_fileobj=weights_path,
    path_in_repo="governed_evo2_weights.pt",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload TOPO-governed Evo 2 fine-tuned weights and invariant anchors",
    token=HF_TOKEN
)

print(f"Successfully deployed! View your model at: https://huggingface.co/{repo_id}")

Ensuring repository exists: frankmorales2020/governed-evo2-7b-topo
Uploading governed artifact ./governed_evo2_final_model/governed_evo2_weights.pt to Hugging Face Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../governed_evo2_weights.pt:   0%|          |  565kB / 13.2GB            

Successfully deployed! View your model at: https://huggingface.co/frankmorales2020/governed-evo2-7b-topo


In [1]:
import torch
from huggingface_hub import hf_hub_download
from evo2 import Evo2

from warnings import filterwarnings
filterwarnings('ignore')

# 1. Initialize fresh base Evo 2 manager
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Initializing official Evo 2 Manager for evo2_7b on {device}...")
evo_manager = Evo2('evo2_7b')
model = getattr(evo_manager, 'model', evo_manager)

# 2. Download your governed artifact directly from Hugging Face Hub
repo_id = "frankmorales2020/governed-evo2-7b-topo"
filename = "governed_evo2_weights.pt"

print(f"Downloading governed weights from Hugging Face Hub ({repo_id})...")
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename)

# 3. Load the checkpoint and restore state
print("Restoring governed weights and TOPO invariant anchors...")
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
anchors = checkpoint["topo_anchors"]
print(f"Successfully loaded TOPO anchors for layers: {list(anchors.keys())}")

# 4. Run inference test
test_prompt = "ACGTTGCAATGCGATCG"
input_seqs = [test_prompt]

print(f"Running inference test with prompt: '{test_prompt}'")
generated_output = evo_manager.generate(
    input_seqs,
    n_tokens=30,
    temperature=0.7,
    top_k=4
)

print("\nInference Output from HF Loaded Governed Model:")
print(generated_output)

Initializing official Evo 2 Manager for evo2_7b on cuda:0...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt




  0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 32/32 [00:00<00:00, 160.59it/s]


Extra keys in state_dict: {'blocks.9.projections._extra_state', 'blocks.2.projections._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.9.mixer.mixer.filter.t', 'blocks.0.projections._extra_state', 'blocks.17.mixer.dense._extra_state', 'blocks.23.projections._extra_state', 'blocks.15.projections._extra_state', 'blocks.27.projections._extra_state', 'blocks.1.projections._extra_state', 'blocks.3.mixer.attn._extra_state', 'blocks.12.projections._extra_state', 'blocks.16.mixer.mixer.filter.t', 'blocks.8.projections._extra_state', 'blocks.20.projections._extra_state', 'blocks.5.projections._extra_state', 'blocks.4.projections._extra_state', 'blocks.13.projections._extra_state', 'blocks.25.projections._extra_state', 'blocks.10.mixer.attn._extra_state', 'blocks.31.mixer.dense._extra_state', 'blocks.18.projections._extra_state', 'blocks.29.projections._extra_state', 'blocks.13.mixer.mixer.filter.t', 'blocks.23.mixer.mixer.filter.t', 'blocks.17.mixer.attn._extra_state', 'blocks.27.mixer.